# Train DeblurGAN on Colab

This notebook trains the DeblurGAN generator (Kupyn et al., "DeblurGAN: Blind
Motion Deblurring Using Conditional Adversarial Networks," CVPR 2018) implemented
in this project's `ml-model/` folder, using a Colab GPU runtime.

**Before running anything below:**

1. Make sure you're on a GPU runtime: `Runtime > Change runtime type > T4 GPU`
   (or better).
2. Get this project's `ml-model/` folder onto this Colab instance. Any of:
   - Zip your local `ml-model/` folder and upload + unzip it here (`Files` pane,
     or `!unzip ml-model.zip -d /content/`), or
   - Put it on Google Drive and mount Drive (done for you in a cell below), or
   - `git clone` the repository if it's pushed somewhere Colab can reach.
3. Adjust and run the `%cd` cell below so it points at wherever `ml-model/`
   ended up -- every relative path used later (`data/`, `weights/`, `train.py`,
   ...) depends on this.

This notebook does **not** run inference or serve the API -- it only trains.
See `README.md` for running `inference.py` / the FastAPI service afterward,
either back on your own machine or here in Colab.


In [ ]:
# Change this to wherever you placed/unzipped/cloned the `ml-model` folder.
%cd /content/ml-model
!pwd
!ls


In [ ]:
# Colab's default runtime already ships a CUDA-enabled PyTorch build, so this
# mainly installs the packages torch doesn't cover here: fastapi, uvicorn,
# pillow, tqdm, python-multipart, etc. There's no need to reinstall torch itself.
!pip install -r requirements.txt


In [ ]:
# Mount Google Drive so checkpoints survive Colab runtime resets/timeouts
# (a Colab session can be recycled mid-training otherwise, losing anything
# not saved somewhere persistent).
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/deblurgan_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to', CHECKPOINT_DIR)


In [ ]:
# Downloads and extracts the GOPRO_Large dataset into data/GOPRO_Large/.
# This is several GB, so expect this to take a while. If it fails (source
# link moved, etc.), see the fallback links printed by the script, or in
# README.md / scripts/download_dataset.py's docstring.
!python scripts/download_dataset.py


In [ ]:
# Adjust --epochs / --batch-size / --n-critic / etc. as needed. A Colab GPU
# (T4/A100) can usually handle a larger --batch-size than the paper's batch
# size of 1 -- 8 is a reasonable starting point on a T4; increase if you have
# more VRAM and reduce if you hit an out-of-memory error.
!python train.py \
    --data-dir data/GOPRO_Large \
    --epochs 300 \
    --batch-size 8 \
    --checkpoint-dir /content/drive/MyDrive/deblurgan_checkpoints


## After training

Your trained generator weights are at:

```
/content/drive/MyDrive/deblurgan_checkpoints/generator_latest.pth
```

(plus a `generator_epoch<N>.pth` snapshot every `--save-every` epochs, and
`train_state_latest.pth` if you need to `--resume` training later.)

To use them for inference, copy that file back into this project as
`weights/generator.pth` -- e.g. download it from Drive, or from within Colab:

```python
import shutil
shutil.copy(
    '/content/drive/MyDrive/deblurgan_checkpoints/generator_latest.pth',
    'weights/generator.pth',
)
```

Then `inference.py` and the FastAPI service (`api/app.py`) will pick it up
automatically -- either run them here in Colab, or copy `generator.pth` down
to your own machine's `ml-model/weights/` folder to run inference locally
(inference does not need a GPU; only training does).
